In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from csgp.layers.kernels import LaplaceL1Kernel
from csgp.design_class import HyperbolicCrossDesign
from csgp.chol_inv import mk_chol_inv

In [2]:
L = 3 # Level-L dyadic grid: m=2^L-1 grid points

dyadic_design = HyperbolicCrossDesign(dyadic_sort=True, return_neighbors=True)(deg=L, input_lb=0, input_ub=1)
design_points = dyadic_design.points.reshape(-1, 1)  # [m, 1] size tensor
print(f'Dyadic sorted design points: {dyadic_design.points}')

Dyadic sorted design points: tensor([0.5000, 0.2500, 0.7500, 0.1250, 0.3750, 0.6250, 0.8750])


In [3]:
import torch

x = torch.tensor([0.35, 0.65])

In [4]:
###########################################################
# PART 1: Use Cholesky decomposition phi(x) = k(x,U)L^{-T}
###########################################################
chol_inv = mk_chol_inv(
    dyadic_design=dyadic_design,
    markov_kernel=LaplaceL1Kernel(lengthscale=1.),
    upper=True)  # [m, m] size tensor
k_xu = LaplaceL1Kernel(lengthscale=1.)(x, design_points)
phi = torch.matmul(k_xu, chol_inv)
print(f"x: {x}")
print(f"phi(x): {phi}")

x: tensor([0.3500, 0.6500])
phi(x): tensor([[ 8.6071e-01,  3.7387e-01, -1.4539e-07, -5.6704e-09,  2.8185e-01,
          1.5002e-07, -3.2482e-08],
        [ 8.6071e-01, -1.4539e-07,  3.7387e-01, -3.2482e-08,  2.0258e-07,
          2.8185e-01, -5.6704e-09]])


In [5]:
def dyadic_nonzero_indices(x: torch.Tensor, L: int, return_anchor=False):
        if x.ndim == 0:
            x = x.unsqueeze(0)  # promote scalar to shape (1,)

        # 2^s for s=1..L
        pow2 = torch.pow(2, torch.arange(1, L+1, device=x.device, dtype=torch.int64))  # (L,)

        # k_s = ceil(2^s * x) clamped to [1, 2^s - 1]
        ks = torch.ceil(x[..., None] * pow2.to(x.dtype)).to(torch.int64)  # (..., L)
        ks = torch.clamp(ks, min=1)
        ks_max = (pow2 - 1)  # (L,)
        ks = torch.minimum(ks, ks_max)  # (..., L)

        # r_s^(odd): force to be odd (right endpoint index made odd)
        # if ks even -> ks-1, else ks
        rs = ks - ((ks & 1) == 0).to(torch.int64)  # (..., L), odd in {1,3,...,2^s-1}

        # position within level s: t_s in {1,...,2^{s-1}}
        ts = (rs + 1) // 2  # (..., L)

        # offsets: number of columns before level s (0-based indexing)
        offsets = (pow2 // 2) - 1  # (L,)

        # global 0-based indices: J_s = offset(s) + (t_s - 1)
        idx = offsets + (ts - 1)  # (..., L)
        
        if return_anchor:
            anchor = torch.tensor([2**L-1, 2**L], device=x.device, dtype=torch.int64).expand((*idx.shape[:-1], 2))  # (..., 2)
            idx = torch.cat([idx, anchor], dim=-1)  # (..., L+2)

        return idx
    
nonzero_idx = dyadic_nonzero_indices(x, L)
print(f"Non-zero indices:\n {nonzero_idx}")
print(f"idx shape: {nonzero_idx.shape}")

nonzero_idx_anchor = dyadic_nonzero_indices(x, L, return_anchor=True)
print(f"Non-zero indices with anchor:\n {nonzero_idx_anchor}")
print(f"idx with anchor shape: {nonzero_idx_anchor.shape}")

Non-zero indices:
 tensor([[0, 1, 4],
        [0, 2, 5]])
idx shape: torch.Size([2, 3])
Non-zero indices with anchor:
 tensor([[0, 1, 4, 7, 8],
        [0, 2, 5, 7, 8]])
idx with anchor shape: torch.Size([2, 5])


In [6]:
###########################################################
# PART 2: Use compact support and sparse psi
###########################################################
import numpy as np

def anchor_points(x: torch.Tensor, ell_c: float = 1.0):
        x = torch.exp(- (x / ell_c)) + torch.exp(- ((1 - x) / ell_c))
        coeff = torch.tensor([1.0 / np.sqrt(2.0 * (1 + np.exp(- 1.0 / ell_c))), 1.0 / np.sqrt(2.0 * (1 - np.exp(- 1.0 / ell_c)))], device=x.device, dtype=x.dtype)
        res = x.unsqueeze(-1) @ coeff.unsqueeze(0)  # (..., 1)
        return res  # (..., 2)
    
def dyadic_psi(x: torch.Tensor, L: int, sigma: float = 1.0, ell_c: float = 1.0, return_anchor: bool = False):
    """
    Batch-wise dyadic nonzero indices.

    Args
    ----
    x : (...,) tensor with values in [0, 1].
        Works with any number of leading batch dims.
    L : int, number of dyadic levels (total columns m = 2^L - 1).

    Returns
    -------
    idx : (..., L) long tensor
        0-based global column indices in dyadic order for each level (DC is level 1).
        The returned shape matches the leading shape of x, with an extra trailing dim of size L.
    """
    if x.ndim == 0:
        x = x.unsqueeze(0)  # promote scalar to shape (1,)
    device, dtype = x.device, x.dtype

    # 2^s for s=1..L
    pow2 = torch.pow(2, torch.arange(1, L+1, device=device, dtype=torch.int64))  # (L,)

    # k_s = ceil(2^s * x) clamped to [1, 2^s - 1]
    ks = torch.ceil(x[..., None] * pow2.to(x.dtype)).to(torch.int64)  # (..., L)
    ks = torch.clamp(ks, min=1)
    ks_max = (pow2 - 1)  # (L,)
    ks = torch.minimum(ks, ks_max)  # (..., L)

    # r_s^(odd): force to be odd (right endpoint index made odd)
    # if ks even -> ks-1, else ks
    rs = ks - ((ks & 1) == 0).to(torch.int64) # (..., L), odd in {1,3,...,2^s-1}

    # position within level s: t_s in {1,...,2^{s-1}}
    ts = (rs + 1) // 2  # (..., L)

    # offsets: number of columns before level s (0-based indexing)
    offsets = (pow2 // 2) - 1  # (L,)

    # global 0-based indices: J_s = offset(s) + (t_s - 1)
    idx = offsets + (ts - 1)  # (..., L)

    # u = HyperbolicCrossDesign(dyadic_sort=True, return_neighbors=True)(deg=L, input_lb=0, input_ub=1).points # (2^L-1,)
    # view_shape = (1,) * x.dim() + (u.shape[0],)      # (1,1,...,1, 2^L-1)
    # u_selected = torch.gather(u.view(view_shape).expand(*x.shape, -1), dim=-1, index=idx) # (..., L)
    # delta = torch.abs(x.unsqueeze(-1) - u_selected) # |x - m2^{-l}|

    delta = torch.abs(x.unsqueeze(-1) - rs/pow2)  # (..., L)
    pow2_f = (1.0 / pow2).to(x.dtype)

    psi =  sigma * torch.sqrt(2 / torch.sinh(pow2_f * 2 * ell_c)) * torch.sinh(ell_c * (pow2_f - delta))
    
    if return_anchor:
        anchor_idx = torch.tensor([2**L-1, 2**L], device=x.device, dtype=torch.int64).expand((*idx.shape[:-1], 2))  # (..., 2)
        idx = torch.cat([idx, anchor_idx], dim=-1)  # (..., L+2)
        anchor_vals = anchor_points(x, ell_c=ell_c)  # (..., 2)
        psi = torch.cat([psi, anchor_vals], dim=-1)  # (...,

    return psi, idx  # (..., L)

In [7]:
# Let's find out the non-zero indices using wavelet design
# Compared to phi_dense, the selected nonzero indices are correct
psi, idx = dyadic_psi(x, L)
print(f"Non-zero idx:\n {idx}")

phi_nonzero = torch.gather(phi, dim=-1, index=idx)
print(f"Non-zero phi(x):\n {phi_nonzero}")
print(f"psi(x):\n {psi}")

psi_anchor, idx_anchor = dyadic_psi(x, L, return_anchor=True)
print(f"Non-zero idx with anchor:\n {idx_anchor}")
print(f"psi(x) with anchor:\n {psi_anchor}")

Non-zero idx:
 tensor([[0, 1, 4],
        [0, 2, 5]])
Non-zero phi(x):
 tensor([[0.8607, 0.3739, 0.2818],
        [0.8607, 0.3739, 0.2818]])
psi(x):
 tensor([[0.4660, 0.2950, 0.2818],
        [0.4660, 0.2950, 0.2818]])
Non-zero idx with anchor:
 tensor([[0, 1, 4, 7, 8],
        [0, 2, 5, 7, 8]])
psi(x) with anchor:
 tensor([[0.4660, 0.2950, 0.2818, 0.7417, 1.0910],
        [0.4660, 0.2950, 0.2818, 0.7417, 1.0910]])


In [8]:
###########################################################
# PART3: verify the batch-wise computing of psi(x)
###########################################################
x = torch.tensor([[0.1, 0.8], [0.8, 0.1], [0.35, 0.65], [0.65, 0.35]]) # (B=4, N=2)
psi, idx = dyadic_psi(x, L)
print(f"Non-zero idx:\n {idx}")
print(f"idx shape: {idx.shape}")
print(f"psi shape: {psi.shape}")

psi_anchor, idx_anchor = dyadic_psi(x, L, return_anchor=True)
print(f"idx shape with anchor: {idx_anchor.shape}")
print(f"psi shape with anchor: {psi_anchor.shape}")

Non-zero idx:
 tensor([[[0, 1, 3],
         [0, 2, 6]],

        [[0, 2, 6],
         [0, 1, 3]],

        [[0, 1, 4],
         [0, 2, 5]],

        [[0, 2, 5],
         [0, 1, 4]]])
idx shape: torch.Size([4, 2, 3])
psi shape: torch.Size([4, 2, 3])
idx shape with anchor: torch.Size([4, 2, 5])
psi shape with anchor: torch.Size([4, 2, 5])


In [9]:
def dyadic_to_dense(vals, idx, m):
    """
    Convert dyadic sparse representation to dense.

    Args
    ----
    vals : (..., fsize, m) tensor
        Values at the dyadic non-zero indices.
    idx : (..., fsize, m) long tensor
        0-based global column indices in dyadic order for each level (DC is level 1).
        The leading shape of idx must match that of vals.

    Returns
    -------
    dense : (..., m) tensor
        Dense representation.
    """
    dense = torch.zeros(*vals.shape[:-1], m, device=vals.device, dtype=vals.dtype)
    dense.scatter_(-1, idx, vals)
    return dense

In [10]:
psi_dense = dyadic_to_dense(psi, idx, 2**L-1)
print(f"psi_dense:\n {psi_dense}")
print(f"psi_dense shape: {psi_dense.shape}")

psi_dense_anchor = dyadic_to_dense(psi_anchor, idx_anchor, 2**L+1)
print(f"psi_dense shape with anchor: {psi_dense_anchor.shape}")

psi_dense:
 tensor([[[0.1307, 0.1962, 0.0000, 0.2818, 0.0000, 0.0000, 0.0000],
         [0.2627, 0.0000, 0.3944, 0.0000, 0.0000, 0.0000, 0.1407]],

        [[0.2627, 0.0000, 0.3944, 0.0000, 0.0000, 0.0000, 0.1407],
         [0.1307, 0.1962, 0.0000, 0.2818, 0.0000, 0.0000, 0.0000]],

        [[0.4660, 0.2950, 0.0000, 0.0000, 0.2818, 0.0000, 0.0000],
         [0.4660, 0.0000, 0.2950, 0.0000, 0.0000, 0.2818, 0.0000]],

        [[0.4660, 0.0000, 0.2950, 0.0000, 0.0000, 0.2818, 0.0000],
         [0.4660, 0.2950, 0.0000, 0.0000, 0.2818, 0.0000, 0.0000]]])
psi_dense shape: torch.Size([4, 2, 7])
psi_dense shape with anchor: torch.Size([4, 2, 9])


In [11]:
import numpy as np

def anchor_points(x: torch.Tensor, ell_c: float = 1.0):
        x = torch.exp(- (x / ell_c)) + torch.exp(- ((1 - x) / ell_c))
        coeff = torch.tensor([1.0 / np.sqrt(2.0 * (1 + np.exp(- 1.0 / ell_c))), 1.0 / np.sqrt(2.0 * (1 - np.exp(- 1.0 / ell_c)))], device=x.device, dtype=x.dtype)
        res = x.unsqueeze(-1) @ coeff.unsqueeze(0)  # (..., 1)
        return res  # (..., 2)
    
x = torch.Tensor(2, 3)
print(x)
res = anchor_points(x)
print(res)
print(res.shape)

tensor([[3.0779e-34, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00]])
tensor([[[0.8270, 1.2166],
         [0.8270, 1.2166],
         [0.8270, 1.2166]],

        [[0.8270, 1.2166],
         [0.8270, 1.2166],
         [0.8270, 1.2166]]])
torch.Size([2, 3, 2])
